# __Project: Mandarin ASR with wav2vec 2.0 and CTC__

- This project follows the general workflow of the Hugging Face tutorial [Fine-Tuning XLS-R for Multilingual ASR with 🤗 Transformers](https://huggingface.co/blog/fine-tune-xlsr-wav2vec2). 

- The original tutorial has been adapted to the current Hugging Face Transformers API and to Mandarin speech recognition using the AISHELL-3 corpus.

- The goal of the project is to fine-tune a pretrained XLS-R model for Mandarin automatic speech recognition (ASR) using Connectionist Temporal Classification (CTC).

Three output representations are investigated:

1. Chinese characters (Hanzi)
2. Tone-numbered Pinyin syllables
3. Tone-numbered Pinyin letters

- This notebook covers the second part of the project, including vocabulary construction, CTC tokenizer construction, feature-extractor configuration, and processor serialization.

- _AI-assisted development was used for code explanation, debugging, and API migration. Python version 3.13.14, Transformers version 5.12.1_

In [67]:
# Standard library
import json
import re
from pathlib import Path

# Environment information
import torch
import torchcodec
import transformers

# Dataset
from datasets import load_from_disk

# Tokenizer and processor
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

print("PyTorch:", torch.__version__)
print("TorchCodec:", torchcodec.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.15.0
TorchCodec: 0.17.0
Transformers: 5.12.1
CUDA available: False


### __📚Start to Build the CTC Vocab__
- including one Hanzi vocab, one Pinyin vocab and one alphabetical phone vocab.

In [68]:
# get the processed dataset root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "processed_data").exists():
    candidate = (
        PROJECT_ROOT
        / "semester-project-chinese-ASR"
    )

    if (candidate / "processed_data").exists():
        PROJECT_ROOT = candidate
    else:
        raise FileNotFoundError(
            "Cannot find processed_data directory."
        )

DATA_ROOT = PROJECT_ROOT / "aishell3"
PROCESSED_DATA_DIR = PROJECT_ROOT / "processed_data"

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_DIR)

Project root: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR
Processed data: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/processed_data


In [69]:
# already have dataset, laod the dataset
dataset = load_from_disk(PROCESSED_DATA_DIR / "aishell3_dataset")

print("Dataset loaded from disk:", dataset)

Dataset loaded from disk: DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio_path', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio_path', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio_path', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})


#### Now We Start to Build CTC Vocab. 
- `[PAD]` is used as both the padding token and CTC blank token.
- `[UNK]` represents an out-of-vocabulary token.
- `|` represents a boundary originating from a space in the transcription.

- __Hanzi__: each token contains every character

- __Pinyin__: each token contains pinyin+tone together

- __Letter__: in alphabet a to z, also includes tone 1 to 5 (Tone numbers 1–4 represent lexical tones,
while tone 5 represents the neutral tone.).

---
##### __🀄️ Hanzi Vocab__

In [70]:
# getting the unique characters in the Hanzi set
hanzi_characters = sorted({
    character
    for text in dataset["train"]["hanzi"]
    for character in text
})

# vocab to id
hanzi_vocab = {
    character: index
    for index, character in enumerate(hanzi_characters)
}

# adding special tokens for unknown and padding
hanzi_vocab["[UNK]"] = len(hanzi_vocab)
hanzi_vocab["[PAD]"] = len(hanzi_vocab)
hanzi_vocab["|"] = len(hanzi_vocab)

print("Hanzi vocabulary size:", len(hanzi_vocab))
print(list(hanzi_vocab.items())[:20])

Hanzi vocabulary size: 4015
[('一', 0), ('丁', 1), ('七', 2), ('万', 3), ('丈', 4), ('三', 5), ('上', 6), ('下', 7), ('不', 8), ('与', 9), ('丐', 10), ('丑', 11), ('专', 12), ('且', 13), ('丕', 14), ('世', 15), ('丘', 16), ('丙', 17), ('业', 18), ('丛', 19)]


---
##### __📦 Pinyin Vocab__

In [71]:
# getting the pinyin pairs from the phone_set in the dataset
def read_train_pinyin_tokens(phone_set_path):
    pinyin_tokens = []
    reading_train_section = False

# read the file, which contains all the pinyin pairs
    with Path(phone_set_path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line == "# The phones frequencies of the train set.":
                reading_train_section = True
                continue

            if line == "# The phones frequencies of the test set.":
                break

            if not reading_train_section:
                continue

            if not line or line.startswith("#"):
                continue

            pinyin_token, frequency = line.split()

            if not re.fullmatch(r"[a-z]+[1-5]", pinyin_token):
                raise ValueError(
                    f"Invalid Pinyin token: {pinyin_token}"
                )

            pinyin_tokens.append(pinyin_token)

    if len(pinyin_tokens) != len(set(pinyin_tokens)):
        raise ValueError("Duplicate Pinyin tokens found in training section.")

    return sorted(pinyin_tokens)

In [72]:
pinyin_tokens = read_train_pinyin_tokens(DATA_ROOT / "phone_set.txt")
pinyin_vocab = {token: index for index, token in enumerate(pinyin_tokens)}

pinyin_vocab["|"] = len(pinyin_vocab)
pinyin_vocab["[UNK]"] = len(pinyin_vocab)
pinyin_vocab["[PAD]"] = len(pinyin_vocab)

print("Pinyin vocabulary size:", len(pinyin_vocab))
print(list(pinyin_vocab.items())[:20])

Pinyin vocabulary size: 1847
[('a1', 0), ('a2', 1), ('a3', 2), ('a4', 3), ('a5', 4), ('ai1', 5), ('ai2', 6), ('ai3', 7), ('ai4', 8), ('ai5', 9), ('an1', 10), ('an2', 11), ('an3', 12), ('an4', 13), ('ang1', 14), ('ang2', 15), ('ang4', 16), ('ao1', 17), ('ao2', 18), ('ao3', 19)]


---
##### __🔤 Letter Vocab__

In [73]:
# getting the vocab directly from the alphabet and numbers
letter_tokens = list("abcdefghijklmnopqrstuvwxyz12345")

letter_vocab = {
    token: index
    for index, token in enumerate(letter_tokens)
}

letter_vocab["|"] = len(letter_vocab)
letter_vocab["[UNK]"] = len(letter_vocab)
letter_vocab["[PAD]"] = len(letter_vocab)

print("Letter vocabulary size:", len(letter_vocab))
print(letter_vocab)

Letter vocabulary size: 34
{'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, '1': 26, '2': 27, '3': 28, '4': 29, '5': 30, '|': 31, '[UNK]': 32, '[PAD]': 33}


In [74]:
vocab_names = ["hanzi", "pinyin", "letter"]

vocab_list = [hanzi_vocab, pinyin_vocab, letter_vocab]

VOCAB_DIR = (PROJECT_ROOT / "vocab")
VOCAB_DIR.mkdir(parents=True, exist_ok=True)

for vocab_name, vocabulary in zip(vocab_names, vocab_list):
    vocab_path = VOCAB_DIR / f"{vocab_name}_vocab.json"

    with vocab_path.open("w", encoding="utf-8") as vocab_file:
        json.dump(
            vocabulary,
            vocab_file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"Saved {vocab_name} vocabulary: "
        f"{vocab_path} "
        f"({len(vocabulary)} tokens)"
    )

Saved hanzi vocabulary: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/vocab/hanzi_vocab.json (4015 tokens)
Saved pinyin vocabulary: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/vocab/pinyin_vocab.json (1847 tokens)
Saved letter vocabulary: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/vocab/letter_vocab.json (34 tokens)


#### __⚙️ Start to Build the CTC Tokenizer__

__CTC Tokenization Strategies__

- The Hanzi, Pinyin-character, and Pinyin-syllable vocabularies use different tokenization units. Therefore, they cannot all use the default tokenization behavior in the same way.

- _Hanzi Tokenization and Letter Tokenization_

    Each Chinese character is treated as an individual token. For example:

    ```text
    - 你好 
    → ["你", "好"]

    - ni3 hao3
    → ["n", "i", "3", "|", "h", "a", "o", "3"]

- _Pinyin Tokenization_
    - The character-level Pinyin representation has a small output vocabulary, but produces longer label sequences. 
    - The syllable-level representation has a larger vocabulary, but produces shorter and more linguistically meaningful label sequences.

In [75]:
# get the vocab file paths for the CTC tiokenizers
hanzi_vocab_path = VOCAB_DIR / "hanzi_vocab.json"
pinyin_vocab_path = VOCAB_DIR / "pinyin_vocab.json"
letter_vocab_path = VOCAB_DIR / "letter_vocab.json"

for vocab_path in [
    hanzi_vocab_path,
    pinyin_vocab_path,
    letter_vocab_path,
]:
    assert vocab_path.exists(), (
        f"Vocabulary not found: "
        f"{vocab_path.resolve()}"
    )

In [76]:
# create the CTC tokenizers for Hanzi, Pinyin, and Letter
hanzi_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(hanzi_vocab_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

letter_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(letter_vocab_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

print("Hanzi Tokenizer size:", len(hanzi_tokenizer))
print("Letter Tokenizer size:", len(letter_tokenizer))

Hanzi Tokenizer size: 4017
Letter Tokenizer size: 36


In [77]:
class PinyinSyllableCTCTokenizer(
    Wav2Vec2CTCTokenizer
):
    def prepare_for_tokenization(
        self,
        text,
        **kwargs,
    ):
        text = "|".join(
            text.lower().split()
        )

        return text, kwargs

pinyin_tokenizer = (
    PinyinSyllableCTCTokenizer(
        vocab_file=str(
            pinyin_vocab_path
        ),
        unk_token="[UNK]",
        pad_token="[PAD]",
        word_delimiter_token="|",
    )
)

print("Pinyin Tokenizer size:", len(pinyin_tokenizer))

Pinyin Tokenizer size: 1849


In [78]:
# Sanity check for the Hanzi tokenizer
hanzi_example = dataset["train"][0]["hanzi"]

hanzi_encoding = hanzi_tokenizer(hanzi_example)

hanzi_tokens = (hanzi_tokenizer.convert_ids_to_tokens(hanzi_encoding.input_ids))

decoded_text = hanzi_tokenizer.decode(
    hanzi_encoding.input_ids,
    group_tokens=False,
)

print("Text:", hanzi_example)
print("IDs:", hanzi_encoding.input_ids)
print("Tokens:", hanzi_tokens)
print("Decoded:", decoded_text)

assert hanzi_tokens == list(hanzi_example)
assert decoded_text == hanzi_example
assert (
    hanzi_tokenizer.unk_token_id
    not in hanzi_encoding.input_ids
)

print("Hanzi tokenizer sanity check passed.")

Text: 但这一点已经足够表明卷福的超高人气
IDs: [151, 3512, 0, 2161, 1053, 2739, 3418, 775, 3199, 1600, 440, 2560, 2410, 3413, 3936, 93, 1879]
Tokens: ['但', '这', '一', '点', '已', '经', '足', '够', '表', '明', '卷', '福', '的', '超', '高', '人', '气']
Decoded: 但这一点已经足够表明卷福的超高人气
Hanzi tokenizer sanity check passed.


In [79]:
# Check and print the tokenization results
pinyin_example_1 = "er2 zi5 wanr2 shou3 ji1"
pinyin_example_2 = "er2zi5wanr2shou3ji1"

tokenizer_list = [pinyin_tokenizer, letter_tokenizer]

for tokenizer in tokenizer_list:
    
    encoding_1 = tokenizer(pinyin_example_1)
    encoding_2 = tokenizer(pinyin_example_2)

    tokens_1 = tokenizer.convert_ids_to_tokens(
        encoding_1.input_ids
    )
    tokens_2 = tokenizer.convert_ids_to_tokens(
        encoding_2.input_ids
    )

    decoded_1 = tokenizer.decode(
        encoding_1.input_ids,
        group_tokens=False,
    )
    decoded_2 = tokenizer.decode(
        encoding_2.input_ids,
        group_tokens=False,
    )

    print(
        f"===== {tokenizer.__class__.__name__} ====="
    )
    print("Input 1:", pinyin_example_1)
    print("Token IDs 1:", encoding_1.input_ids)
    print("Tokens 1:", tokens_1)
    print("Decoded 1:", decoded_1)

    print()

    print("Input 2:", pinyin_example_2)
    print("Token IDs 2:", encoding_2.input_ids)
    print("Tokens 2:", tokens_2)
    print("Decoded 2:", decoded_2)

    print()

===== PinyinSyllableCTCTokenizer =====
Input 1: er2 zi5 wanr2 shou3 ji1
Token IDs 1: [371, 1844, 1816, 1844, 1511, 1844, 1326, 1844, 620]
Tokens 1: ['er2', '|', 'zi5', '|', 'wanr2', '|', 'shou3', '|', 'ji1']
Decoded 1: er2 zi5 wanr2 shou3 ji1

Input 2: er2zi5wanr2shou3ji1
Token IDs 2: [371, 1816, 1511, 1326, 620]
Tokens 2: ['er2', 'zi5', 'wanr2', 'shou3', 'ji1']
Decoded 2: er2zi5wanr2shou3ji1

===== Wav2Vec2CTCTokenizer =====
Input 1: er2 zi5 wanr2 shou3 ji1
Token IDs 1: [4, 17, 27, 31, 25, 8, 30, 31, 22, 0, 13, 17, 27, 31, 18, 7, 14, 20, 28, 31, 9, 8, 26]
Tokens 1: ['e', 'r', '2', '|', 'z', 'i', '5', '|', 'w', 'a', 'n', 'r', '2', '|', 's', 'h', 'o', 'u', '3', '|', 'j', 'i', '1']
Decoded 1: er2 zi5 wanr2 shou3 ji1

Input 2: er2zi5wanr2shou3ji1
Token IDs 2: [4, 17, 27, 25, 8, 30, 22, 0, 13, 17, 27, 18, 7, 14, 20, 28, 9, 8, 26]
Tokens 2: ['e', 'r', '2', 'z', 'i', '5', 'w', 'a', 'n', 'r', '2', 's', 'h', 'o', 'u', '3', 'j', 'i', '1']
Decoded 2: er2zi5wanr2shou3ji1



---
After creating the tokenizer and vocanb, we can combine the audio and the tokenized text together

### __🧪 Processor, combined audio and text together__

- Why Data Processor?

    _Raw audio → Feature Extractor → input_values_

    _Text → Tokenizer → labels_

    Wav2Vec2Processor combines these two components into one consistent interface.

In [80]:
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16_000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

hanzi_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=hanzi_tokenizer,
)

pinyin_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=pinyin_tokenizer,
)

letter_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=letter_tokenizer,
)

In [81]:
processors = {
    "hanzi": hanzi_processor,
    "pinyin": pinyin_processor,
    "letter": letter_processor,
}

for name, processor in (
    processors.items()
):
    print(
        f"{name}: "
        f"{len(processor.tokenizer)} tokens, "
        f"{processor.feature_extractor.sampling_rate} Hz"
    )

hanzi: 4017 tokens, 16000 Hz
pinyin: 1849 tokens, 16000 Hz
letter: 36 tokens, 16000 Hz


In [82]:
PROCESSOR_DIR = (PROJECT_ROOT / "processors")
PROCESSOR_DIR.mkdir(parents=True, exist_ok=True)

In [83]:
hanzi_processor.save_pretrained(PROCESSOR_DIR / "hanzi")

pinyin_processor.save_pretrained(PROCESSOR_DIR / "pinyin")

letter_processor.save_pretrained(PROCESSOR_DIR / "letter")

print("Processors saved to:", PROCESSOR_DIR.resolve())

Processors saved to: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/processors
